# OmniGenome - A Demonstration based on RNA Secondary Structure Prediction
GitHub: https://github.com/yangheng95/OmniGenome
OmniGenome Hub: Huggingface Spaces

## Introduction
OmniGenome is a comprehensive package designed for pretrained genomic foundation models (FMs) development and FM benchmark. 
OmniGenome have the following key features:
- Automated genomic FM benchmarking on public genomic datasets
- Scalable genomic FM training and fine-tuning on genomic tasks
- Diversified genomic FMs implementation
- Easy-to-use pipeline for genomic FM development with no coding expertise required
- Accessible OmniGenome Hub for sharing FMs, datasets, and pipelines
- Extensive documentation and tutorials for genomic FM development

We begin to introduce OmniGenome by delivering a demonstration to train a model to predict RNA secondary structures. The dataset used in this demonstration is the bpRNA dataset which contains RNA sequences and their corresponding secondary structures. The secondary structure of an RNA sequence is a set of base pairs that describe the folding of the RNA molecule. The secondary structure of an RNA sequence is important for understanding the function of the RNA molecule. In this demonstration, we will train a model to predict the secondary structure of an RNA sequence given its primary sequence.

## Requirements
OmniGenome requires the following recommended dependencies:
- Python 3.9+
- PyTorch 2.0.0+
- Transformers 4.37.0+
- Pandas 1.3.3+
- Others in case of specific tasks

## Fine-tuning Genomic FMs for RNA Secondary Structure Prediction

### Step 1: Import Libraries

In [1]:
import os

import autocuda
import torch
from metric_visualizer import MetricVisualizer

from omnigenome import OmniGenomeDatasetForTokenClassification
from omnigenome import ClassificationMetric
from omnigenome import OmniSingleNucleotideTokenizer
from omnigenome import OmniGenomeModelForTokenClassification
from omnigenome import Trainer



                                
   **  +----------- **           ___                     _ 
  @@                 @@         / _ \  _ __ ___   _ __  (_)
 @@* #============== *@@       | | | || '_ ` _ \ | '_ \ | |
 @@*                 *@@       | |_| || | | | | || | | || |
 *@@  +------------ *@@         \___/ |_| |_| |_||_| |_||_|
  *@*               @@*       
   *@@  #========= @@*        
    *@@*         *@@*          
      *@@  +---@@@*              ____  
        *@@*   **               / ___|  ___  _ __    ___   _ __ ___    ___ 
          **@**                | |  _  / _ \| '_ \  / _ \ | '_ ` _ \  / _ \ 
        *@@* *@@*              | |_| ||  __/| | | || (_) || | | | | ||  __/ 
      *@@ ---+  @@*             \____| \___||_| |_| \___/ |_| |_| |_| \___| 
    *@@*         *@@*          
   *@@ =========#  @@*         
  *@@               @@*        
 *@@ -------------+  @@*        ____                      _   
 @@                   @@       | __ )   ___  _ __    ___ | |__ 
 

### Step 2: Define and Initialize the Tokenizer

In [2]:
# Predefined dataset label mapping
label2id = {"(": 0, ")": 1, ".": 2}

# The is FM is exclusively powered by the OmniGenome package
model_name_or_path = "anonymous8/OmniGenome-186M"

# Generally, we use the tokenizers from transformers library, such as AutoTokenizer
# tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

# However, OmniGenome provides specialized tokenizers for genomic data, such as single nucleotide tokenizer and k-mers tokenizer
# we can force the tokenizer to be used in the model
tokenizer = OmniSingleNucleotideTokenizer.from_pretrained(model_name_or_path)

### Step 3: Define and Initialize the Model

In [3]:
# We have implemented a diverse set of genomic models in OmniGenome, please refer to the documentation for more details
ssp_model = OmniGenomeModelForTokenClassification(
    model_name_or_path,
    tokenizer=tokenizer,
    label2id=label2id,
)

You are using a model of type omnigenome to instantiate a model of type mprna. This is not supported for all configurations of models and can yield errors.
Some weights of OmniGenomeModel were not initialized from the model checkpoint at anonymous8/OmniGenome-186M and are newly initialized: ['OmniGenome.pooler.dense.bias', 'OmniGenome.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[2025-02-10 00:21:41] [OmniGenome 0.2.0alpha]  Model Name: OmniGenomeModelForTokenClassification
Model Metadata: {'library_name': 'OmniGenome', 'omnigenome_version': '0.2.0alpha', 'torch_version': '2.1.2+cu12.1+gita8e7c98cb95ff97bb30a728c6b2a1ce6bff946eb', 'transformers_version': '4.48.3', 'model_cls': 'OmniGenomeModelForTokenClassification', 'tokenizer_cls': 'OmniSingleNucleotideTokenizer', 'model_name': 'OmniGenomeModelForTokenClassification'}
Base Model Name: anonymous8/OmniGenome-186M
Model Type: omnigenome
Model Architecture: None
Model Parameters: 185.886801 M
Model Config: OmniGenomeConfig {
  "OmniGenomefold_config": null,
  "_name_or_path": "anonymous8/OmniGenome-186M",
  "attention_probs_dropout_prob": 0.0,
  "auto_map": {
    "AutoConfig": "anonymous8/OmniGenome-186M--configuration_omnigenome.OmniGenomeConfig",
    "AutoModel": "anonymous8/OmniGenome-186M--modeling_omnigenome.OmniGenomeModel",
    "AutoModelForMaskedLM": "anonymous8/OmniGenome-186M--modeling_omnigenome.OmniG

### Step 4: Define and Load the Dataset

In [4]:
# necessary hyperparameters
epochs = 10
learning_rate = 2e-5
weight_decay = 1e-5
batch_size = 4
max_length = 512
seeds = [45]  # Each seed will be used for one run


# Load the dataset according to the path
train_file = "toy_datasets/Archive2/train.json"
test_file = "toy_datasets/Archive2/test.json"
valid_file = "toy_datasets/Archive2/valid.json"

train_set = OmniGenomeDatasetForTokenClassification(
    data_source=train_file,
    tokenizer=tokenizer,
    label2id=label2id,
    max_length=max_length,
)
test_set = OmniGenomeDatasetForTokenClassification(
    data_source=test_file,
    tokenizer=tokenizer,
    label2id=label2id,
    max_length=max_length,
)
valid_set = OmniGenomeDatasetForTokenClassification(
    data_source=valid_file,
    tokenizer=tokenizer,
    label2id=label2id,
    max_length=max_length,
)
train_loader = torch.utils.data.DataLoader(
    train_set, batch_size=batch_size, shuffle=True
)
valid_loader = torch.utils.data.DataLoader(valid_set, batch_size=batch_size)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=batch_size)

[2025-02-10 00:21:41] [OmniGenome 0.2.0alpha]  Detected max_length=512 in the dataset, using it as the max_length.
[2025-02-10 00:21:41] [OmniGenome 0.2.0alpha]  Loading data from toy_datasets/Archive2/train.json...
[2025-02-10 00:21:41] [OmniGenome 0.2.0alpha]  Loaded 608 examples from toy_datasets/Archive2/train.json
[2025-02-10 00:21:41] [OmniGenome 0.2.0alpha]  Detected shuffle=True, shuffling the examples...


100%|██████████████████████████████████████████████████████████████████████████████| 608/608 [00:00<00:00, 2448.08it/s]


[2025-02-10 00:21:41] [OmniGenome 0.2.0alpha]  Max sequence length updated -> Reset max_length=504, label_padding_length=504
[2025-02-10 00:21:41] [OmniGenome 0.2.0alpha]  {'avg_seq_len': 130.54276315789474, 'max_seq_len': 501, 'min_seq_len': 56, 'avg_label_len': 504.0, 'max_label_len': 504, 'min_label_len': 504}
[2025-02-10 00:21:41] [OmniGenome 0.2.0alpha]  Preview of the first two samples in the dataset:
[2025-02-10 00:21:41] [OmniGenome 0.2.0alpha]  {'input_ids': tensor([0, 6, 5, 9, 9, 6, 6, 6, 6, 4, 5, 4, 4, 9, 4, 6, 5, 4, 4, 6, 4, 9, 6, 6,
        4, 4, 5, 5, 4, 5, 5, 9, 6, 4, 9, 5, 5, 5, 9, 9, 9, 5, 5, 6, 4, 4, 5, 9,
        5, 4, 6, 4, 4, 6, 9, 6, 4, 4, 4, 5, 6, 9, 9, 9, 9, 4, 6, 5, 6, 5, 5, 6,
        4, 9, 6, 6, 9, 4, 6, 9, 6, 9, 6, 9, 5, 4, 6, 9, 9, 6, 6, 5, 4, 9, 6, 9,
        6, 4, 6, 4, 6, 9, 4, 6, 6, 9, 5, 4, 9, 5, 5, 5, 5, 4, 4, 6, 5, 9, 2, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

100%|████████████████████████████████████████████████████████████████████████████████| 82/82 [00:00<00:00, 2556.44it/s]


[2025-02-10 00:21:41] [OmniGenome 0.2.0alpha]  Max sequence length updated -> Reset max_length=328, label_padding_length=328
[2025-02-10 00:21:41] [OmniGenome 0.2.0alpha]  {'avg_seq_len': 131.23170731707316, 'max_seq_len': 321, 'min_seq_len': 67, 'avg_label_len': 328.0, 'max_label_len': 328, 'min_label_len': 328}
[2025-02-10 00:21:41] [OmniGenome 0.2.0alpha]  Preview of the first two samples in the dataset:
[2025-02-10 00:21:41] [OmniGenome 0.2.0alpha]  {'input_ids': tensor([0, 6, 6, 5, 5, 5, 5, 6, 9, 6, 6, 9, 6, 9, 4, 6, 9, 9, 6, 6, 9, 9, 4, 4,
        5, 4, 5, 4, 5, 5, 5, 6, 5, 5, 9, 6, 9, 5, 4, 5, 6, 9, 6, 6, 6, 4, 6, 4,
        9, 5, 6, 5, 6, 6, 6, 9, 9, 5, 6, 4, 6, 9, 5, 5, 5, 6, 9, 5, 6, 6, 6, 6,
        5, 5, 6, 5, 5, 4, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

100%|████████████████████████████████████████████████████████████████████████████████| 76/76 [00:00<00:00, 2749.37it/s]


[2025-02-10 00:21:41] [OmniGenome 0.2.0alpha]  Max sequence length updated -> Reset max_length=312, label_padding_length=312
[2025-02-10 00:21:41] [OmniGenome 0.2.0alpha]  {'avg_seq_len': 117.39473684210526, 'max_seq_len': 308, 'min_seq_len': 60, 'avg_label_len': 312.0, 'max_label_len': 312, 'min_label_len': 312}
[2025-02-10 00:21:41] [OmniGenome 0.2.0alpha]  Preview of the first two samples in the dataset:
[2025-02-10 00:21:41] [OmniGenome 0.2.0alpha]  {'input_ids': tensor([0, 6, 4, 5, 9, 5, 5, 4, 9, 6, 6, 5, 5, 4, 4, 6, 9, 9, 6, 6, 9, 9, 4, 4,
        6, 6, 5, 6, 9, 6, 5, 6, 4, 5, 9, 6, 9, 9, 4, 4, 9, 5, 6, 5, 4, 4, 6, 4,
        9, 5, 6, 9, 6, 4, 6, 9, 9, 5, 4, 4, 5, 5, 5, 9, 5, 4, 5, 9, 6, 6, 6, 6,
        9, 5, 6, 5, 5, 4, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

### Step 5: Define the Metrics
We have implemented a diverse set of genomic metrics in OmniGenome, please refer to the documentation for more details.
Users can also define their own metrics by inheriting the `OmniGenomeMetric` class. 
The `compute_metrics` can be a metric function list and each metric function should return a dictionary of metrics.

In [5]:
compute_metrics = [
    ClassificationMetric(ignore_y=-100).accuracy_score,
    ClassificationMetric(ignore_y=-100, average="macro").f1_score,
    ClassificationMetric(ignore_y=-100).matthews_corrcoef,
]


## Step 6: Define and Initialize the Trainer

In [6]:
# Initialize the MetricVisualizer for logging the metrics
mv = MetricVisualizer(name="OmniGenome-186M-SSP")

for seed in seeds:
    optimizer = torch.optim.AdamW(
        ssp_model.parameters(), lr=learning_rate, weight_decay=weight_decay
    )
    trainer = Trainer(
        model=ssp_model,
        train_loader=train_loader,
        eval_loader=valid_loader,
        test_loader=test_loader,
        batch_size=batch_size,
        epochs=epochs,
        optimizer=optimizer,
        compute_metrics=compute_metrics,
        seeds=seed,
        device=autocuda.auto_cuda(),
    )

    # metrics = trainer.train()
    # test_metrics = metrics["test"][-1]
    # mv.log(model_name_or_path.split("/")[-1], "F1", test_metrics["f1_score"])
    # mv.log(
    #     model_name_or_path.split("/")[-1],
    #     "Accuracy",
    #     test_metrics["accuracy_score"],
    # )
    # print(metrics)
    # mv.summary()

### Step 7. Experimental Results Visualization
The experimental results are visualized in the following plots. The plots show the F1 score and accuracy of the model on the test set for each run. The average F1 score and accuracy are also shown.

|### Step 8. Model Checkpoint for Sharing
The model checkpoint can be saved and shared with others for further use. The model checkpoint can be loaded using the following code:

**Regular checkpointing and resuming are good practices to save the model at different stages of training.**

In [7]:
path_to_save = "OmniGenome-186M-SSP"
ssp_model.save(path_to_save, overwrite=True)

# Load the model checkpoint
ssp_model = ssp_model.load(path_to_save)
results = ssp_model.inference("CAGUGCCGAGGCCACGCGGAGAACGAUCGAGGGUACAGCACUA")
print(results["predictions"])
print("logits:", results["logits"])

[2025-02-10 00:21:43] [OmniGenome 0.2.0alpha]  The model is saved to OmniGenome-186M-SSP.
[2025-02-10 00:21:43] [OmniGenome 0.2.0alpha]  Warning: The value of the key torch_dtype in the loaded model is torch.float16, but the current value is float16.
[2025-02-10 00:21:43] [OmniGenome 0.2.0alpha]  Warning: The value of the key _name_or_path in the loaded model is OmniGenome-186M-SSP, but the current value is anonymous8/OmniGenome-186M.
[2025-02-10 00:21:43] [OmniGenome 0.2.0alpha]  Warning: The value of the key _commit_hash in the loaded model is None, but the current value is 689e0c46403fb44d0c56ac3846bec87ae5768d1e.
[2025-02-10 00:21:43] [OmniGenome 0.2.0alpha]  Warning: The value of the key transformers_version in the loaded model is 4.48.3, but the current value is 4.41.0.dev0.
[2025-02-10 00:21:43] [OmniGenome 0.2.0alpha]  Warning: The value of the key model_type in the loaded model is mprna, but the current value is omnigenome.


C:\Users\chuan\miniconda3\lib\site-packages\transformers\tokenization_utils_base.py:2699: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


[')', '.', '(', '.', '.', '.', '(', '.', ')', ')', '(', '(', '.', ')', '.', '(', '(', ')', ')', ')', ')', '.', '.', '(', ')', '(', '(', '.', '.', '.', '.', '.', ')', '.', ')', '.', '(', '.', '.', '.', '.', ')', ')']
logits: tensor([[0.3016, 0.3544, 0.3440],
        [0.3297, 0.3490, 0.3212],
        [0.3392, 0.2206, 0.4402],
        [0.3874, 0.2345, 0.3781],
        [0.3462, 0.2267, 0.4271],
        [0.3258, 0.2344, 0.4398],
        [0.3676, 0.2067, 0.4257],
        [0.3703, 0.3570, 0.2728],
        [0.3386, 0.2695, 0.3919],
        [0.3200, 0.4203, 0.2597],
        [0.3192, 0.4343, 0.2465],
        [0.3737, 0.2737, 0.3525],
        [0.4032, 0.2471, 0.3496],
        [0.3187, 0.2633, 0.4180],
        [0.3509, 0.3817, 0.2674],
        [0.3688, 0.2332, 0.3979],
        [0.4601, 0.2886, 0.2513],
        [0.4961, 0.2819, 0.2221],
        [0.3373, 0.3633, 0.2994],
        [0.3717, 0.4301, 0.1983],
        [0.3336, 0.3783, 0.2882],
        [0.3317, 0.4158, 0.2525],
        [0.2709, 0.3509, 0.3


# What if someone doesn't know how to initialize the model?

In [8]:
# We can load the model checkpoint using the ModelHub
from omnigenome import ModelHub

ssp_model = ModelHub.load("OmniGenome-186M-SSP")
results = ssp_model.inference("CAGUGCCGAGGCCACGCGGAGAACGAUCGAGGGUACAGCACUA")
print(results["predictions"])
print("logits:", results["logits"])

[2025-02-10 00:21:46] [OmniGenome 0.2.0alpha]  Model Name: OmniGenomeModelForTokenClassification
Model Metadata: {'library_name': 'OmniGenome', 'omnigenome_version': '0.2.0alpha', 'torch_version': '2.1.2+cu12.1+gita8e7c98cb95ff97bb30a728c6b2a1ce6bff946eb', 'transformers_version': '4.48.3', 'model_cls': 'OmniGenomeModelForTokenClassification', 'tokenizer_cls': 'OmniSingleNucleotideTokenizer', 'model_name': 'OmniGenomeModelForTokenClassification'}
Base Model Name: OmniGenome-186M-SSP
Model Type: mprna
Model Architecture: ['OmniGenomeModel']
Model Parameters: 185.886801 M
Model Config: OmniGenomeConfig {
  "OmniGenomefold_config": null,
  "_attn_implementation_autoset": true,
  "_name_or_path": "OmniGenome-186M-SSP",
  "architectures": [
    "OmniGenomeModel"
  ],
  "attention_probs_dropout_prob": 0.0,
  "auto_map": {
    "AutoConfig": "anonymous8/OmniGenome-186M--configuration_omnigenome.OmniGenomeConfig",
    "AutoModel": "anonymous8/OmniGenome-186M--modeling_omnigenome.OmniGenomeModel"

## Step 8. Model Inference

In [9]:
examples = [
    "GCUGGGAUGUUGGCUUAGAAGCAGCCAUCAUUUAAAGAGUGCGUAACAGCUCACCAGC",
    "AUCUGUACUAGUUAGCUAACUAGAUCUGUAUCUGGCGGUUCCGUGGAAGAACUGACGUGUUCAUAUUCCCGACCGCAGCCCUGGGAGACGUCUCAGAGGC",
]

results = ssp_model.inference(examples)
structures = ["".join(prediction) for prediction in results["predictions"]]
print(results)
print(structures)

{'predictions': [['.', '(', '.', '.', '.', ')', '.', '(', '(', '(', '.', '(', '(', '(', '(', ')', ')', ')', ')', ')', ')', '(', '.', '.', '.', '.', '.', '(', '.', '.', '.', ')', ')', ')', ')', ')', '(', '(', '(', '(', ')', ')', ')', '(', ')', ')', '(', '.', ')', '.', '.', '.', ')', ')', '(', '.', ')', ')'], [')', ')', ')', '(', ')', '(', ')', '(', '(', '.', '(', '(', '(', '.', ')', ')', '.', '.', '.', ')', '.', ')', '.', '.', '.', ')', '(', ')', ')', ')', ')', ')', ')', ')', '.', '.', '.', '.', '.', ')', ')', '.', '.', ')', '.', '.', '.', '(', '(', '.', '.', '(', ')', ')', '.', '.', ')', '(', '.', '.', '.', '.', ')', '(', ')', '.', '.', '.', '.', '.', ')', ')', '.', '.', ')', '.', ')', '(', '.', '.', '.', '(', '(', '(', '(', '(', ')', ')', ')', ')', '.', ')', ')', '.', '.', ')', ')', '.', ')', '.']], 'logits': tensor([[[0.3103, 0.3713, 0.3184],
         [0.3386, 0.2700, 0.3914],
         [0.4116, 0.2157, 0.3726],
         [0.3499, 0.2299, 0.4204],
         [0.3254, 0.2426, 0.4321],
   

### Step 9. Pipeline Creation
The OmniGenome package provides pipelines for genomic FM development. The pipeline can be used to train, fine-tune, and evaluate genomic FMs. The pipeline can be used with a single command to train a genomic FM on a dataset. The pipeline can also be used to fine-tune a pre-trained genomic FM on a new dataset. The pipeline can be used to evaluate the performance of a genomic FM on a dataset. The pipeline can be used to generate predictions using a genomic FM.

In [10]:
# from omnigenome import Pipeline, PipelineHub
# 
# pipeline = Pipeline(
#     name="OmniGenome-186M-SSP-Pipeline",
#     # model_name_or_path="OmniGenome-186M-SSP",  # The model name or path can be specified
#     # tokenizer="OmniGenome-186M-SSP",  # The tokenizer can be specified
#     model_name_or_path=ssp_model,
#     tokenizer=ssp_model.tokenizer,
#     datasets={
#         "train": "toy_datasets/train.json",
#         "test": "toy_datasets/test.json",
#         "valid": "toy_datasets/valid.json",
#     },
#     trainer=trainer,
#     device=ssp_model.model.device,
# )

### Using the Pipeline

In [11]:
# results = pipeline(examples[0])
# print(results)
# 
# pipeline.train()
# 
# pipeline.save("OmniGenome-186M-SSP-Pipeline", overwrite=True)
# 
# pipeline = PipelineHub.load("OmniGenome-186M-SSP-Pipeline")
# results = pipeline(examples)
# print(results)

## Web Demo for RNA Secondary Structure Prediction

In [12]:
import numpy as np
import json
import gradio as gr
import RNA
from omnigenome import ModelHub
import os
print(os.listdir('.'))
ssp_model = ModelHub.load("OmniGenome-186M-SSP")

def ss_validity_loss(rna_strct):
    dotCount = 0
    leftCount = 0
    rightCount = 0
    unmatched_positions = []  # 用于记录未匹配括号的位置
    uncoherentCount = 0
    prev_char = ""
    for i, char in enumerate(rna_strct):
        if prev_char != char:
            uncoherentCount += 1
        prev_char = char

        if char == "(":
            leftCount += 1
            unmatched_positions.append(i)  # 记录左括号位置
        elif char == ")":
            if leftCount > 0:
                leftCount -= 1
                unmatched_positions.pop()  # 移除最近的左括号位置
            else:
                rightCount += 1
                unmatched_positions.append(i)  # 记录右括号位置
        elif char == ".":
            dotCount += 1
        else:
            raise ValueError(f"Invalid character {char} in RNA structure")
    match_loss = (leftCount + rightCount) / (len(rna_strct) - dotCount + 1e-5)
    return match_loss


def find_invalid_ss_positions(rna_strct):
    left_brackets = []  # 存储左括号的位置
    right_brackets = []  # 存储未匹配的右括号的位置
    for i, char in enumerate(rna_strct):
        if char == "(":
            left_brackets.append(i)
        elif char == ")":
            if left_brackets:
                left_brackets.pop()  # 找到匹配的左括号，从列表中移除
            else:
                right_brackets.append(i)  # 没有匹配的左括号，记录右括号的位置
    return left_brackets + right_brackets


def fold(rna_sequence):
    try:
        ref_struct = RNA.fold(rna_sequence)[0]
        RNA.svg_rna_plot(rna_sequence, ref_struct, f"real_structure.svg")
    
        pred_structure = "".join(ssp_model.inference(rna_sequence)["predictions"])
        print(pred_structure)
        if ss_validity_loss(pred_structure) == 0:
            RNA.svg_rna_plot(rna_sequence, pred_structure, f"predicted_structure.svg")
            pred_structure, _ = repair_rna_structure(
                rna_sequence, pred_structure
            )
            return (
                ref_struct,
                pred_structure,
                "real_structure.svg",
                "predicted_structure.svg",
            )
        else:
            # return blank image of predicted structure
            # generate a blank svg image
            with open("predicted_structure.svg", "w") as f:
                f.write(
                    '<svg xmlns="http://www.w3.org/2000/svg" width="100" height="100"></svg>'
                )
            return (
                ref_struct,
                pred_structure,
                "real_structure.svg",
                "best_pred_struct.svg",
            )
    except Exception as e:
        with open("real_structure.svg", "w") as f:
            f.write(
                '<svg xmlns="http://www.w3.org/2000/svg" width="100" height="100"></svg>'
            )
        with open("predicted_structure.svg", "w") as f:
            f.write(
                '<svg xmlns="http://www.w3.org/2000/svg" width="100" height="100"></svg>'
            )
        return e, e, "real_structure.svg", "predicted_structure.svg"


def repair_rna_structure(rna_sequence, invalid_struct):
    try:
        invalid_ss_positions = find_invalid_ss_positions(invalid_struct)
        for pos_idx in invalid_ss_positions:
            if invalid_struct[pos_idx] == "(":
                invalid_struct = (
                    invalid_struct[:pos_idx] + "." + invalid_struct[pos_idx + 1 :]
                )
            else:
                invalid_struct = (
                    invalid_struct[:pos_idx] + "." + invalid_struct[pos_idx + 1 :]
                )

        best_pred_struct = invalid_struct
        RNA.svg_rna_plot(rna_sequence, best_pred_struct, f"best_pred_struct.svg")
        return best_pred_struct, "best_pred_struct.svg"
    except Exception as e:
        with open("best_pred_struct.svg", "w") as f:
            f.write(
                '<svg xmlns="http://www.w3.org/2000/svg" width="100" height="100"></svg>'
            )
        return e, "best_pred_struct.svg"


def sample_rna_sequence():
    example = examples[np.random.randint(0, len(examples))]
    RNA.svg_rna_plot(example["seq"], example["label"], f"annotated_structure.svg")

    return example["seq"], example["label"], "annotated_structure.svg"


# 在gr.Blocks中添加CSS样式
with gr.Blocks(css="""
.center-image {
    display: flex;
    justify-content: center;
    width: 100%;
}
.center-image img {
    max-width: 100%;
    height: auto;
}
    """) as demo:
    gr.Markdown("<h1 style='text-align: center;'>RNA Secondary Structure Prediction</h1>")
    gr.Markdown(
        "<p style='text-align: center;'>This demo uses the OmniGenome foundation model to predict RNA secondary structure.</p>"
    )


    with gr.Row():
        with gr.Row():
            rna_input = gr.Textbox(
                label="RNA Sequence", placeholder="Enter RNA sequence here..."
            )
        with gr.Row():
            strcut_input = gr.Textbox(
                label="Annotated Secondary Structure",
                placeholder="Enter RNA secondary structure here...",
            )

    with gr.Row():
        #     examples = [
        #     ["GCGUCACACCGGUGAAGUCGCGCGUCACACCGGUGAAGUCGC"],
        #     ["GCUGGGAUGUUGGCUUAGAAGCAGCCAUCAUUUAAAGAGUGCGUAACAGCUCACCAGCGCUGGGAUGUUGGCUUAGAAGCAGCCAUCAUUUAAAGAGUGCGUAACAGCUCACCAGC"],
        #     ["GGCUGGUCCGAGUGCAGUGGUGUUUACAACUAAUUGAUCACAACCAGUUACAGAUUUCUUUGUUCCUUCUCCACUCCCACUGCUUCACUUGACUAGCCUU"],
        # ]
        #     gr.Examples(examples=examples, label="Examples", inputs=[rna_input])
        with open("toy_datasets/Archive2/test.json", "r") as f:
            examples = []
            for line in f:
                examples.append(json.loads(line))

        sample_button = gr.Button("Sample a RNA Sequence from RNAStrand2 testset")

    with gr.Row():
        submit_button = gr.Button("Run Prediction")

    with gr.Row():
        ref_structure_output = gr.Textbox(
            label="Secondary Structure by ViennaRNA", interactive=False
        )

    with gr.Row():
        pred_structure_output = gr.Textbox(
            label="Secondary Structure by Model", interactive=False
        )

    with gr.Row():
        anno_structure_output = gr.Image(
            label="Annotated Secondary Structure", show_share_button=True
        )
        real_image = gr.Image(
            label="Secondary Structure by ViennaRNA", show_share_button=True
        )
        predicted_image = gr.Image(
            label="Secondary Structure by Model", show_share_button=True
        )

    with gr.Row():
        repair_button = gr.Button("Run Prediction Repair")

    submit_button.click(
        fn=fold,
        inputs=rna_input,
        outputs=[
            ref_structure_output,
            pred_structure_output,
            real_image,
            predicted_image,
        ],
    )

    repair_button.click(
        fn=repair_rna_structure,
        inputs=[rna_input, pred_structure_output],
        outputs=[pred_structure_output, predicted_image],
    )

    sample_button.click(
        fn=sample_rna_sequence, outputs=[rna_input, strcut_input, anno_structure_output]
    )
demo.launch(share=True)

['.gradio', '.ipynb_checkpoints', 'annotated_structure.svg', 'benchmark', 'best_pred_struct.svg', 'easy_rna_design_emoo.py', 'eterna100_vienna2.txt', 'EternaV2_RNA_design_demo.py', 'extract_evolution_history.ffi.py', 'jack.ffi', 'jic.ffi', 'OmniGenome-186M-SSP', 'predicted_structure.svg', 'process_data.py', 'readme.md', 'real_structure.svg', 'secondary_structure_prediction_demo.ipynb', 'solved_sequences.json', 'solved_sequences_legacy.json', 'tmp_ckpt_20250209_233944_d261fb157b6e01e03f1f9d495527fc3b6ea98624e765aac116f2b6e9d69b790f.pt', 'toy_datasets', 'true_struct.svg', 'tutorials', 'web_rna_design.py', 'zero_shot_secondary_structure_prediction.py']
[2025-02-10 00:21:50] [OmniGenome 0.2.0alpha]  Model Name: OmniGenomeModelForTokenClassification
Model Metadata: {'library_name': 'OmniGenome', 'omnigenome_version': '0.2.0alpha', 'torch_version': '2.1.2+cu12.1+gita8e7c98cb95ff97bb30a728c6b2a1ce6bff946eb', 'transformers_version': '4.48.3', 'model_cls': 'OmniGenomeModelForTokenClassification

C:\Users\chuan\miniconda3\lib\site-packages\gradio\routes.py:644: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")
C:\Users\chuan\miniconda3\lib\site-packages\fastapi\applications.py:4495: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  return self.router.on_event(event_type)


Running on local URL:  http://127.0.0.1:7860


C:\Users\chuan\miniconda3\lib\site-packages\gradio\analytics.py:93: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if StrictVersion(latest_pkg_version) > StrictVersion(current_pkg_version):


IMPORTANT: You are using gradio version 3.50.0, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://66066175f8a39b5ccc.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


### Conclusion
In this demonstration, we have shown how to fine-tune a genomic foundation model for RNA secondary structure prediction using the OmniGenome package. We have also shown how to use the trained model for inference and how to create a web demo for RNA secondary structure prediction. We hope this demonstration will help you get started with genomic foundation model development using OmniGenome.